# edge3/edge4 — 10年検定ランナー（E3A–E3D）

事前登録: `docs/26_preregistration_edge3_edge4.md`（**LOCKED**, N=4, α=0.0125）。

**規律（厳守）**
- 拘束力ある判定は**この10年実データ実行**で確定。短期の好成績は楽観側の上限としてのみ扱う。
- 数値は各 runner が出す **JSON を単一スカラで直読**して転記（表示破損のまま転記しない）。
- 当たるまで候補を増やさない／ルールを後出しで変えない。全REJECTなら「3本目なし」と正直に結論。

実行順: ①Drive → ②設定 → ③依存 → ④データ取得 → ⑤4本実行 → ⑥サマリ。

## ① Google Drive をマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ② パス設定（ここだけ環境に合わせて編集）

- `REPO_DIR`  : このリポジトリ（research/ がある場所）。GitHub から clone するか Drive 上のパス。
- `DATA_DIR`  : 10年CSV の置き場（④で自動取得して埋める）。
- `REPORTS_DIR`: 結果 JSON の出力先（Drive 推奨＝消えない）。

In [ ]:
import os, sys, subprocess

# --- 編集する3行 ---
REPO_DIR    = '/content/chien-monitor'
DATA_DIR    = '/content/drive/MyDrive/edge_research/data'
REPORTS_DIR = '/content/drive/MyDrive/edge_research/reports'
GIT_URL     = 'https://github.com/iq87jun-star/chien-monitor.git'
GIT_BRANCH  = 'claude/new-session-7rAJ3'
# --------------------

if not os.path.isdir(os.path.join(REPO_DIR, 'research')):
    print('cloning', GIT_URL)
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, GIT_URL, REPO_DIR], check=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
research_dir = os.path.join(REPO_DIR, 'research')
assert os.path.isdir(research_dir), f'research/ が無い: {REPO_DIR}'
if research_dir not in sys.path:
    sys.path.insert(0, research_dir)
os.environ['EDGE_DATA_DIR'] = DATA_DIR
os.environ['EDGE_REPORTS_DIR'] = REPORTS_DIR
# costs.csv / policy_rates テンプレを repo から DATA_DIR へ（無ければ）
import shutil
for f in ['costs.csv', 'policy_rates_monthly.csv']:
    src, dst = os.path.join(research_dir, 'data', f), os.path.join(DATA_DIR, f)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst); print('copied', f)
print('OK  repo =', REPO_DIR, '\n    data =', DATA_DIR, '\n    reports =', REPORTS_DIR)

## ③ 依存導入

In [ ]:
!pip -q install numpy pandas scipy dukascopy-python

## ④ データ取得（Dukascopy → DATA_DIR）

FX13＋指数3 の H1 を取得し `v7_weekly_returns.csv` を生成。**10年×16銘柄は数十分**かかる。
既存ファイルはスキップ（再開可）。短期で試すなら `--start/--end` を狭めて先に動作確認。

⚠️ `policy_rates_monthly.csv` は E3C 用に**実値を手動で**埋める（テンプレが入っている）。

In [ ]:
fetch = os.path.join(research_dir, 'fetch_data.py')
buildv7 = os.path.join(research_dir, 'build_v7_weekly.py')
# 本番は --start 2016-01-01 --end 2025-12-31。まず動作確認したいなら下を短期に。
!python3 "{fetch}" --start 2016-01-01 --end 2025-12-31 --data-dir "{DATA_DIR}"
!EDGE_DATA_DIR="{DATA_DIR}" python3 "{buildv7}"
print('\nDATA_DIR files:', sorted(os.listdir(DATA_DIR)))

## ⑤ 4本を実行（E3A → E3B → E3C → E3D）

env を import 前に確定させるため `data_io` と各 runner を毎回 reload する。

In [ ]:
import importlib, json, traceback

RUNNERS = [
    ('edge3a_eqmr_10y',    'edge3a_result.json', 'E3A 株価指数MR'),
    ('edge3b_leadlag_10y', 'edge3b_result.json', 'E3B リードラグ'),
    ('edge3c_carry_10y',   'edge3c_result.json', 'E3C 横断キャリー(spot)'),
    ('edge3d_session_10y', 'edge3d_result.json', 'E3D セッションBO'),
]
results = {}
for mod_name, out_json, label in RUNNERS:
    print(f'\n===== {label} ({mod_name}) =====')
    try:
        import data_io; importlib.reload(data_io)
        mod = importlib.import_module(mod_name); importlib.reload(mod)
        mod.main()
        with open(os.path.join(REPORTS_DIR, out_json)) as f:
            results[mod_name] = json.load(f)
    except Exception as e:
        print('FAILED:', e); traceback.print_exc()
        results[mod_name] = {'error': str(e)}
print('\nDONE. JSON は', REPORTS_DIR, 'に保存済み。')

## ⑥ サマリ（このセル出力をそのまま貼ってください）

ADOPT = 全6ゲート通過。LEAD = 純益>0 かつ p<0.10。それ以外 REJECT。

In [ ]:
import pandas as pd
rows = []
for mod_name, out_json, label in RUNNERS:
    r = results.get(mod_name, {})
    if 'error' in r:
        rows.append({'edge': label, 'grade': 'ERROR', 'note': r['error'][:60]}); continue
    g = r.get('gates', {})
    rows.append({
        'edge': label, 'grade': r.get('grade'), 'n': r.get('n_trades'),
        'net_10y': round(r.get('net_profit_10y', float('nan')), 4),
        'p_net': round(r.get('perm_p_net', float('nan')), 4),
        'jk_maxp': round(r.get('jackknife_max_p', float('nan')), 4),
        'corr_v7': round(r.get('corr_v7', float('nan')), 3),
        'gates_pass': sum(bool(v) for v in g.values()),
        'g1g2g3g4g5g6': ''.join('1' if g.get(k) else '0' for k in
            ['g1_profit','g2_perm_p','g3_jackknife','g4_placebo','g5_cost','g6_independent']),
    })
print('alpha = 0.0125 (Bonferroni 0.05/4) | period = 2016-01..2025-12')
print(pd.DataFrame(rows).to_string(index=False))
print('\n--- 生 JSON（転記用・単一スカラ）---')
print(json.dumps(results, ensure_ascii=False, indent=2))